# Decision Planner Comparison


## 1) Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Install Dependencies


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn


## 3) Load Dataset


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
required = ['X_train.npy','X_test.npy','y_train.npy','y_test.npy']
print('Data path:', DATA_PATH)
print('Files:', os.listdir(DATA_PATH))
missing = [f for f in required if not os.path.exists(os.path.join(DATA_PATH,f))]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

X_test=np.load(os.path.join(DATA_PATH,'X_test.npy'))
y_test=np.load(os.path.join(DATA_PATH,'y_test.npy')).reshape(-1)
if X_test.ndim==3 and X_test.shape[-1]==1: X_test=X_test[...,0]
print('X_test:',X_test.shape,'y_test:',y_test.shape)



## 4) Generate Sample Risk Scores


In [ ]:
def make_samples(X,y,n=600,seed=42):
    rng=np.random.default_rng(seed)
    idx=rng.choice(len(y),size=min(n,len(y)),replace=False)
    amap={0:'BENIGN',1:'DOS',2:'PORTSCAN',3:'BRUTEFORCE',4:'DDOS'}
    rows=[]
    for i in idx:
        yi=int(y[i]); atk=amap.get(yi,f'ATTACK_{yi}')
        s=float(np.clip(np.mean(np.abs(X[i])),0,1))
        r=float(np.clip(0.55*s+0.35*(yi>0)+rng.normal(0,0.1),0,1))
        if r>0.85:t='BLOCK_IP'
        elif r>0.65:t='RATE_LIMIT'
        elif r>0.45 and atk!='BENIGN':t='ALERT_ADMIN'
        else:t='ALLOW'
        rows.append({'risk_score':r,'attack_type':atk,'target_action':t,'is_benign':atk=='BENIGN'})
    return pd.DataFrame(rows)
samples=make_samples(X_test,y_test)
samples.head()



## 5) Strategies + Metrics


In [ ]:
A=['BLOCK_IP','RATE_LIMIT','ALERT_ADMIN','ALLOW']
def p1(r,a):
    if r>0.85:return 'BLOCK_IP'
    if r>0.65:return 'RATE_LIMIT'
    if r>0.45:return 'ALERT_ADMIN'
    return 'ALLOW'
def p2(r,a):
    u=a.upper()
    if 'DDOS' in u or 'DOS' in u:return 'BLOCK_IP' if r>0.6 else 'RATE_LIMIT'
    if 'SCAN' in u or 'BRUTE' in u:return 'RATE_LIMIT' if r>0.55 else 'ALERT_ADMIN'
    return 'ALERT_ADMIN' if r>0.7 else 'ALLOW'
def p3(r,a,rng):
    probs=np.array([0.05+0.70*r,0.15+0.35*r,0.35-0.10*r,0.45-0.95*r]); probs=np.clip(probs,1e-6,None); probs/=probs.sum(); return rng.choice(A,p=probs)
def p4(r,a):
    s=3 if r>0.8 else 2 if r>0.6 else 1 if r>0.4 else 0
    q=np.array([[0.10,0.15,0.30,0.70],[0.25,0.45,0.55,0.20],[0.70,0.65,0.50,0.10],[0.95,0.70,0.35,0.05]])
    return A[int(np.argmax(q[s]))]

def eval_method(name,df,seed=42):
    rng=np.random.default_rng(seed); pred=[]; t0=time.perf_counter()
    for _,r in df.iterrows():
        if name=='Risk Threshold Policy':pred.append(p1(float(r.risk_score),r.attack_type))
        elif name=='Rule Based Policy':pred.append(p2(float(r.risk_score),r.attack_type))
        elif name=='Probabilistic Decision Planner':pred.append(p3(float(r.risk_score),r.attack_type,rng))
        else:pred.append(p4(float(r.risk_score),r.attack_type))
    lat=(time.perf_counter()-t0)/len(df)
    pred=np.array(pred)
    acc=float(np.mean(pred==df.target_action.values))
    ben=pred[df.is_benign.values]
    fmr=float(np.mean(ben!='ALLOW')) if len(ben)>0 else 0.0
    return {'Method':name,'DecisionAccuracy':acc,'FalseMitigationRate':fmr,'DecisionLatencySec':lat}

methods=['Risk Threshold Policy','Rule Based Policy','Probabilistic Decision Planner','Reinforcement Learning Agent (Simulated)']
res=pd.DataFrame([eval_method(m,samples) for m in methods])
res



## 6) Comparison Charts


In [ ]:
res.plot(x='Method',y='DecisionAccuracy',kind='bar',figsize=(10,4),ylim=(0,1),title='Decision Accuracy')
plt.grid(axis='y',alpha=0.3); plt.tight_layout(); plt.show()
res.plot(x='Method',y='FalseMitigationRate',kind='bar',figsize=(10,4),ylim=(0,1),title='False Mitigation Rate')
plt.grid(axis='y',alpha=0.3); plt.tight_layout(); plt.show()
res.plot(x='Method',y='DecisionLatencySec',kind='bar',figsize=(10,4),title='Decision Latency')
plt.tight_layout(); plt.show()

